# Development benchmark

Set `SKIN_LESION_REPORT` to the accepted S09 `benchmark-report.json` for measured development results. The loader verifies the report's SHA-256, plan digest and embedded registry, including the model/seed links used below. It does not open and validate the 27 fitted run records or their files. The accepted report is the frozen, unversioned benchmark result under the original metric and calibration contracts. The [current development status](../docs/development-status-2026-09-22.md) explains what was checked separately.

For a data-free illustration, set `SKIN_LESION_MODE=synthetic`. That mode builds generated records with constructed predictions and cannot be cited as experimental evidence. Execute from the repository root with a fresh kernel.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from src.notebook_story import accepted_report, summary_rows, synthetic_report

get_ipython().run_line_magic("matplotlib", "inline")
mode = os.environ.get("SKIN_LESION_MODE", "accepted")
if mode == "accepted":
    report = accepted_report(Path(os.environ["SKIN_LESION_REPORT"]))
    source = "ACCEPTED S09 DEVELOPMENT REPORT"
elif mode == "synthetic":
    report = synthetic_report()
    source = "ILLUSTRATIVE SYNTHETIC DATA, NOT A MEASURED RUN"
else:
    raise ValueError("SKIN_LESION_MODE must be accepted or synthetic")
print(
    source, "| run IDs:", ", ".join(job["run_id"] for job in report["registry"]["jobs"])
)
print(
    "Images:" if mode == "accepted" else "Generated records:",
    report["counts"]["images"],
    "groups:",
    report["counts"]["groups"],
    "class counts:",
    report["counts"]["classes"],
)

## Comparison

ROC-AUC is image-level; the interval resamples lesion/known-duplicate components within class on shared draws. It is conditional on three prespecified seeds and known groups. Seed standard deviation is a different quantity from the 95% bootstrap interval. Average precision is not trapezoidal PR-AUC. The 10:1 error cost is an illustrative policy, not measured clinical benefit.

In [ ]:
rows = summary_rows(report, "roc_auc")
table = pd.DataFrame(rows)
print(source)
print(table.to_string(index=False))
fig, ax = plt.subplots(figsize=(14, max(4, 0.8 * len(rows))))
positions = range(len(rows))
ax.errorbar(
    [r["estimate"] for r in rows],
    positions,
    xerr=[
        [r["estimate"] - r["interval"][0] for r in rows],
        [r["interval"][1] - r["estimate"] for r in rows],
    ],
    fmt="o",
    capsize=3,
)
ax.set_yticks(
    list(positions),
    [r["model"] + "\n" + ", ".join(r["run_ids"]) for r in rows],
    fontsize=7,
)
ax.set_xlabel("Development ROC-AUC (95% grouped bootstrap interval)")
ax.set_title(source)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## Calibration, referral and failures

The accepted report keeps fitted probabilities, sparse reliability bins and locked referral counts by run. A change in the retained set cannot be read as a full-cohort gain. Error counts are descriptive: repeated images across seeds are not new observations. Synthetic mode has no calibration or referral audit.

In [ ]:
if mode == "accepted":
    primary = report["differences"]["efficientnet_full_unweighted"]["roc_auc"]
    sensitivity = report["differences"]["efficientnet_full_unweighted"]["sensitivity"]
    print(
        "Primary unweighted full EfficientNet minus logistic ROC-AUC:",
        primary["estimate"],
        primary["interval"],
    )
    print("Sensitivity difference:", sensitivity["estimate"], sensitivity["interval"])
    selected = next(
        row["run_ids"] for row in rows if row["model"] == "efficientnet_full_unweighted"
    )
    sensitivity_by_seed = report["models"]["efficientnet_full_unweighted"][
        "sensitivity"
    ]["seed_values"]
    specificity_by_seed = report["models"]["efficientnet_full_unweighted"][
        "specificity"
    ]["seed_values"]
    positives = report["counts"]["classes"]["1"]
    negatives = report["counts"]["classes"]["0"]
    for run_id, tpr, tnr in zip(
        selected, sensitivity_by_seed, specificity_by_seed, strict=True
    ):
        print(
            run_id,
            "full-cohort FN/FP:",
            round((1 - tpr) * positives),
            round((1 - tnr) * negatives),
        )
    for run_id in selected:
        section = report["calibration_and_referral"][run_id]["development"]
        print(
            run_id,
            "raw/fitted Brier:",
            section["raw_score"]["brier_score"],
            section["fitted_probability"]["brier_score"],
            "retained/referred/FN/FP:",
            *(
                section["selective"][key]
                for key in (
                    "retained_count",
                    "referred_count",
                    "retained_fn",
                    "retained_fp",
                )
            ),
        )
    run_id = selected[0]
    bins = report["calibration_and_referral"][run_id]["development"][
        "fitted_probability"
    ]["reliability_bins"]
    print(run_id, "fitted reliability bins:")
    print(pd.DataFrame(bins).to_string(index=False))
    occupied = [item for item in bins if item["count"]]
    fig2, ax2 = plt.subplots(figsize=(6, 5))
    ax2.plot([0, 1], [0, 1], linestyle="--", color="gray", label="identity")
    ax2.scatter(
        [item["mean_probability"] for item in occupied],
        [item["positive_fraction"] for item in occupied],
        s=[max(15, item["count"] / 6) for item in occupied],
    )
    ax2.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="Mean fitted probability per bin",
        ylabel="Observed melanoma fraction per bin",
        title=f"{run_id}: development reliability",
    )
    ax2.legend()
    fig2.tight_layout()
    plt.show()
else:
    print(source, ": calibration and referral audit not available")

## Retrospective GMM clipping diagnosis

The accepted GMM ROC-AUC remains the fitted-probability endpoint. A separate read-only analysis of the three saved development runs compared each run's raw GMM scores with its fitted probabilities. The original sigmoid calibration first clipped probabilities to `[1e-10, 1-1e-10]`. Most raw scores fell outside that interval, so distinct inputs became ties before the sigmoid. All three saved slopes were positive, which could not restore the lost ordering.

| Seed | Raw-score ROC-AUC | Accepted fitted ROC-AUC | Scores outside clip, of 1,395 |
| --- | ---: | ---: | ---: |
| 17 | 0.736432 | 0.560258 | 1,292 |
| 42 | 0.731701 | 0.556168 | 1,291 |
| 73 | 0.740983 | 0.594885 | 1,220 |
| Mean | 0.736372 | 0.570437 | not applicable |

These raw-score values diagnose an information loss in the frozen method. They do not replace its endpoint, show that raw probabilities are calibrated, or demonstrate the performance of the [future v2 GMM method](../docs/calibration-v2.md). The analysis uses saved scores; this notebook neither refits a calibrator nor trains a model. See the [diagnostic script](../scripts/diagnose_calibration.py) for the source checks and calculation.

## Limits

The accepted results use development data. Full fine-tuning improved ROC-AUC and specificity against HSV logistic regression in the prespecified contrast; its sensitivity interval crosses zero. GMM's fitted ROC-AUC is affected by the clipping described above, and referral retained missed melanoma images. Patient links and unknown near-duplicates are not ruled out. No diagnosis, clinical safety or deployment claim follows from this notebook. The separate free-Colab full-job check was not completed and supplies none of these runs.